# Udaplay_02 Solution Project
Tool-driven GameAgent with LLM routing/synthesis (with deterministic fallback when no OpenAI key is set).

In [1]:
import os
import json
import logging
from pathlib import Path

os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")
logging.basicConfig(level=logging.INFO)

from vector_db import VectorDB
from agent_tools.game_tools import VectorDBTool, ResultEvaluator, WebSearchTool
from agent_tools.game_agent import GameAgent

PROJECT_DIR = Path('.').resolve()
GAMES_DIR = PROJECT_DIR / 'games'

def load_games():
    games = []
    for path in sorted(GAMES_DIR.glob('*.json')):
        with path.open('r', encoding='utf-8') as f:
            game = json.load(f)
        if 'Name' in game and 'Description' in game:
            games.append(game)
    return games

db = VectorDB(collection_name='games_collection', persist_dir=str(PROJECT_DIR / 'chromadb'))
db.ingest(load_games())

vector_tool = VectorDBTool(db)
evaluator = ResultEvaluator(distance_threshold=0.35)
web_tool = WebSearchTool(api_key=os.getenv('TAVILY_API_KEY'))
agent = GameAgent(
    vector_tool=vector_tool,
    evaluator=evaluator,
    web_tool=web_tool,
    model='gpt-4o-mini',
    llm_api_key=os.getenv('OPENAI_API_KEY'),
)


INFO:root:Initializing VectorDB...


INFO:root:Using collection 'games_collection'.


INFO:root:No new documents to add (all IDs already exist).


In [2]:
queries = [
    'When was Super Mario World released?',
    'Tell me more about the first game you mentioned',
    'Which platform is Minecraft available on?',
]
queries


['When was Super Mario World released?',
 'Which platform is Minecraft available on?',
 'Upcoming VR space shooter in 2026?']

In [3]:
for q in queries:
    result = agent.ask(q, n_results=3)
    print(f'\nQuery: {q}')
    print(f"Source: {result['source']}")
    print(f"Reasoning: {result['reasoning']}")
    print(f"Answer: {result['answer']}")
    if result.get('citations'):
        print('Citations:')
        for c in result['citations']:
            print(f"  [{c['id']}] {c['label']} -> {c['source']}")


INFO:root:Using internal DB retrieval...


INFO:root:[VectorDBTool] Retrieving for query: When was Super Mario World released?


INFO:root:Using internal DB retrieval...


INFO:root:[VectorDBTool] Retrieving for query: Which platform is Minecraft available on?


INFO:root:Using internal DB retrieval...


INFO:root:[VectorDBTool] Retrieving for query: Upcoming VR space shooter in 2026?



Query: When was Super Mario World released?
Source: internal_db
Reasoning: No LLM configured; used threshold-based evaluator.
Answer: Best match: Super Mario World (1990) on Super Nintendo Entertainment System (SNES). A classic platformer where Mario embarks on a quest to save Princess Toadstool and Dinosaur Land from Bowser.

Query: Which platform is Minecraft available on?
Source: internal_db
Reasoning: No LLM configured; used threshold-based evaluator.
Answer: Best match: Gran Turismo 5 (2010) on PlayStation 3. A comprehensive racing simulator featuring a vast selection of vehicles and tracks, with realistic driving physics.

Query: Upcoming VR space shooter in 2026?
Source: internal_db
Reasoning: No LLM configured; used threshold-based evaluator.
Answer: Best match: Marvel's Spider-Man 2 (2023) on PlayStation 5. The sequel to the acclaimed Spider-Man game, featuring both Peter Parker and Miles Morales as playable characters.
